# Advanced Certification in AIML
## A Program by IIIT-H and TalentSprint

In [ ]:
#@title Explanation Video
from IPython.display import HTML

HTML("""<video width="854" and height="480" controls>
  <source src="https://cdn.iiith.talentsprint.com/aiml/Experiment_related_data/Walkthrough/Hackathon_Voice_based.mp4" type="video/mp4">
</video>
""")

# Hackathon: Voice commands based E-commerce ordering system
The goal of the hackathon is to train your model on different types of voice data (such as studio data and your own team data) and able to place order based on user preferences.

## Grading = 40 Marks

### **Objectives:**

Stage 0 - Obtain Features from Audio samples

Stage 1 (22 Marks) - Define and train a CNN model on Studio data and deploy the model in the server

Stage 2 (18 Marks) - Collect your voice samples (team data) and refine the classifier trained on Studio_data. Deploy the model in the server.

## Dataset Description

The data contains voice samples of classes - Zero, One, Two, Three, Four, Five. Each class is denoted by a numerical label from 0 to 5.

The audio files collected in a Studio dataset contain very few noise samples and all the files are in wav format.

The audio files recorded for the studio are saved with the following naming convention:

● Class Representation + user_id + sample_ID (or noise + sample_ID)

> For example: The voice sample by the user b2 recorded “Zero”, it is saved as 0_b2_35.wav. Here 35 is sample ID, 2 is the user id and ‘0’ is the label of that sample.




In [1]:
#@title Please run the setup to download the dataset

from IPython import get_ipython
ipython = get_ipython()

notebook= "Hackathon2 - Voice E-commerce Ordering System" #name of the notebook

def setup():
    ipython.magic("sx wget https://cdn.iiith.talentsprint.com/aiml/Hackathon_data/B17_studio_rev_data.zip")
    ipython.magic("sx unzip B17_studio_rev_data.zip ")
    print ("Setup completed successfully")

setup()

Setup completed successfully


In [2]:
import os
import sys
import glob
import torch
import librosa
import warnings
import numpy as np
import torch.nn as nn
from time import sleep
from torch import optim
import torch.nn.functional as F
from torch.autograd import Variable
warnings.filterwarnings('ignore')

## **Stage 0:** Obtain Features from Audio samples
---

### Generate features from an audio sample of '.wav' format
- Code is available to extract the features

In [3]:
# Caution: Do not change the default parameters
def get_features(filepath, sr=8000, n_mfcc=30, n_mels=128, frames = 15):
    # The following function contains code to produce features of the audio sample.
    y, sr = librosa.load(filepath, sr=sr)
    D = np.abs(librosa.stft(y))**2
    S = librosa.feature.melspectrogram(S=D)
    S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=n_mels)
    log_S = librosa.power_to_db(S,ref=np.max)
    features = librosa.feature.mfcc(S=log_S, n_mfcc=n_mfcc)
    if features.shape[1] < frames :
        features = np.hstack((features, np.zeros((n_mfcc, frames - features.shape[1]))))
    elif features.shape[1] > frames:
        features = features[:, :frames]

    # Find 1st order delta_mfcc
    delta1_mfcc = librosa.feature.delta(features, order=1)

    # Find 2nd order delta_mfcc
    delta2_mfcc = librosa.feature.delta(features, order=2)

    # Stacking delta_mfcc features in sequence horizontally (column wise)
    features = np.hstack((delta1_mfcc.flatten(), delta2_mfcc.flatten()))

    # Increase the dimension by inserting an axis along second dimension
    features = features.flatten()[:,np.newaxis]

    # Convert the numpy.ndarray to a Tensor object
    features = Variable(torch.from_numpy(features)).float()
    return features

All the voice samples needed for training are present in the folder `"studio_data"`

In [4]:
%ls

B17_studio_rev_data.zip  sample_data/  studio_data/


##**Stage 1**:  Define and train a CNN model on Studio data and deploy the model in the server

---


### a) Extract features of Studio data (4 Marks)

 Load 'Studio data' and extract mfcc features

 **Evaluation Criteria:**

 * Complete the code in the load_data function
 * The function should take path of the folder containing audio samples as input
 * It should return features of all the audio samples present in the specified folder into single array (list of lists or 2-d numpy array) and their respective labels should be returned too

In [5]:
def load_data(folder_path):
    """
    Loads all .wav files from folder_path, extracts MFCC-based features
    using get_features(), and pulls the class label out of the filename.

    Filenames look like: 0_b2_35.wav  -> label is the first token ("0")
    """
    features = []
    labels = []

    wav_files = glob.glob(os.path.join(folder_path, '*.wav'))
    print(f"Found {len(wav_files)} wav files in {folder_path}")

    for filepath in wav_files:
        filename = os.path.basename(filepath)
        label = int(filename.split('_')[0])          # class label is the first token
        feat = get_features(filepath)                  # tensor shape (900, 1)
        features.append(feat.numpy().flatten())         # flatten to (900,)
        labels.append(label)

    features = np.array(features)   # shape (N, 900)
    labels = np.array(labels)       # shape (N,)
    return features, labels


Load data from studio_data folder for extracting all features and labels

In [6]:
studio_recorded_features, studio_recorded_labels = load_data('/content/studio_data')

Found 3979 wav files in /content/studio_data


Use train_test_split for splitting the train and test data

In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    studio_recorded_features,
    studio_recorded_labels,
    test_size=0.2,
    random_state=42,
    stratify=studio_recorded_labels
)

print("Train shape:", X_train.shape, "Test shape:", X_test.shape)


Train shape: (3183, 900) Test shape: (796, 900)


Load the dataset with DataLoader
- Refer to [torch.utils.data.TensorDataset](https://pytorch.org/docs/stable/data.html#torch.utils.data.TensorDataset)
- Refer to [torch.utils.data.DataLoader](https://pytorch.org/docs/stable/data.html#torch.utils.data.DataLoader)

In [8]:
from torch.utils.data import TensorDataset, DataLoader

# Conv1d expects (batch, channels, length) -> we treat the 900 features as
# channels and use a sequence length of 1 (matches conv1's in_channels=900)
X_train_tensor = torch.from_numpy(X_train).float().unsqueeze(2)   # (N, 900, 1)
y_train_tensor = torch.from_numpy(y_train).long()

X_test_tensor = torch.from_numpy(X_test).float().unsqueeze(2)     # (N, 900, 1)
y_test_tensor = torch.from_numpy(y_test).long()

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

batch_size = 16
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print("Batches per epoch (train):", len(train_loader))


Batches per epoch (train): 199


### b) Define your CNN architecture (4 Marks)

[Hint](https://pytorch.org/docs/stable/generated/torch.nn.Conv1d.html)

In [9]:
## Define your CNN Architecture
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()

        # Sample Convolution Layer 1
        self.conv1 = nn.Conv1d(in_channels=900, out_channels=400, kernel_size=1)
        self.bn1 = nn.BatchNorm1d(400)
        self.relu1 = nn.ReLU()

        # Sample Maxpool for the Convolutional Layer 1
        self.maxpool1 = nn.MaxPool1d(1)

        # Sample Dropout Layer
        self.dropout = nn.Dropout(p=0.25)

        # Convolutional Layer 2
        self.conv2 = nn.Conv1d(in_channels=400, out_channels=200, kernel_size=1)
        self.bn2 = nn.BatchNorm1d(200)
        self.relu2 = nn.ReLU()
        self.maxpool2 = nn.MaxPool1d(1)

        # Convolutional Layer 3
        self.conv3 = nn.Conv1d(in_channels=200, out_channels=100, kernel_size=1)
        self.bn3 = nn.BatchNorm1d(100)
        self.relu3 = nn.ReLU()
        self.maxpool3 = nn.MaxPool1d(1)

        # Fully Connected Layer -> 6 classes (Zero..Five)
        self.fc1 = nn.Linear(100, 6)
        self.logsoftmax = nn.LogSoftmax(dim=1)

    def forward(self, x):
        # Convolution Layer 1, Maxpool and Dropout
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu1(out)
        out = self.maxpool1(out)
        out = self.dropout(out)

        # Convolution Layer 2, Maxpool and Dropout
        out = self.conv2(out)
        out = self.bn2(out)
        out = self.relu2(out)
        out = self.maxpool2(out)
        out = self.dropout(out)

        # Convolution Layer 3, Maxpool and Dropout
        out = self.conv3(out)
        out = self.bn3(out)
        out = self.relu3(out)
        out = self.maxpool3(out)
        out = self.dropout(out)

        # Flatten (batch, 100, 1) -> (batch, 100)
        out = out.view(out.size(0), -1)

        # Fully connected + LogSoftmax
        out = self.fc1(out)
        out = self.logsoftmax(out)
        return out


In [10]:
# To run the training on GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [11]:
model = Net()
model = model.to(device)
print(model)

criterion = nn.NLLLoss()                                   # pairs with LogSoftmax output
optimizer = optim.Adam(model.parameters(), lr=0.001)        # Adam generally converges fastest here


Net(
  (conv1): Conv1d(900, 400, kernel_size=(1,), stride=(1,))
  (bn1): BatchNorm1d(400, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu1): ReLU()
  (maxpool1): MaxPool1d(kernel_size=1, stride=1, padding=0, dilation=1, ceil_mode=False)
  (dropout): Dropout(p=0.25, inplace=False)
  (conv2): Conv1d(400, 200, kernel_size=(1,), stride=(1,))
  (bn2): BatchNorm1d(200, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu2): ReLU()
  (maxpool2): MaxPool1d(kernel_size=1, stride=1, padding=0, dilation=1, ceil_mode=False)
  (conv3): Conv1d(200, 100, kernel_size=(1,), stride=(1,))
  (bn3): BatchNorm1d(100, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu3): ReLU()
  (maxpool3): MaxPool1d(kernel_size=1, stride=1, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=100, out_features=6, bias=True)
  (logsoftmax): LogSoftmax(dim=1)
)


### c) Train and classify on the studio_data (3 Marks)

The goal here is to train the Model on voice samples collected in studio data and validate it continuously to calculate the loss and accuracy for the train dataset across each epoch.

Iterate over images in the train_loader and perform the following steps.

1. First, zero out the gradients using zero_grad()

2. Pass the data to the model. Convert the data to GPU before passing data  to the model

3. Calculate the loss using a Loss function

4. Perform Backward pass using backward() to update the weights

5. Optimize and predict by using the torch.max()

6. Calculate the accuracy of the train dataset


In [12]:
num_epochs = 30

train_losses = []
train_accuracies = []

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for data, target in train_loader:
        data, target = data.to(device), target.to(device)

        optimizer.zero_grad()               # 1. zero out gradients
        output = model(data)                 # 2. forward pass
        loss = criterion(output, target)     # 3. compute loss
        loss.backward()                      # 4. backward pass
        optimizer.step()                     # 5. update weights

        running_loss += loss.item() * data.size(0)
        _, predicted = torch.max(output, 1)  # 6. get predicted class
        correct += (predicted == target).sum().item()
        total += target.size(0)

    epoch_loss = running_loss / total
    epoch_acc = 100.0 * correct / total
    train_losses.append(epoch_loss)
    train_accuracies.append(epoch_acc)

    print(f"Epoch [{epoch+1}/{num_epochs}]  Loss: {epoch_loss:.4f}  Train Accuracy: {epoch_acc:.2f}%")


Epoch [1/30]  Loss: 1.4127  Train Accuracy: 44.39%
Epoch [2/30]  Loss: 1.0084  Train Accuracy: 62.93%
Epoch [3/30]  Loss: 0.8775  Train Accuracy: 68.39%
Epoch [4/30]  Loss: 0.8084  Train Accuracy: 71.25%
Epoch [5/30]  Loss: 0.7287  Train Accuracy: 74.21%
Epoch [6/30]  Loss: 0.6741  Train Accuracy: 76.56%
Epoch [7/30]  Loss: 0.6247  Train Accuracy: 77.91%
Epoch [8/30]  Loss: 0.5762  Train Accuracy: 79.39%
Epoch [9/30]  Loss: 0.5625  Train Accuracy: 80.33%
Epoch [10/30]  Loss: 0.5422  Train Accuracy: 81.21%
Epoch [11/30]  Loss: 0.5194  Train Accuracy: 81.75%
Epoch [12/30]  Loss: 0.4838  Train Accuracy: 83.29%
Epoch [13/30]  Loss: 0.4957  Train Accuracy: 82.50%
Epoch [14/30]  Loss: 0.4606  Train Accuracy: 84.20%
Epoch [15/30]  Loss: 0.4294  Train Accuracy: 84.32%
Epoch [16/30]  Loss: 0.4348  Train Accuracy: 85.20%
Epoch [17/30]  Loss: 0.4184  Train Accuracy: 85.17%
Epoch [18/30]  Loss: 0.4150  Train Accuracy: 85.11%
Epoch [19/30]  Loss: 0.4007  Train Accuracy: 85.80%
Epoch [20/30]  Loss: 

### d) Testing Evaluation for CNN model (3 Marks)

Evaluate model with the given test data

1. Transform and load the test images.

2. Pass the test data through the model (network) to get the outputs

3. Get the predictions from a maximum value using torch.max

4. Compare with the actual labels and get the count of the correct labels

5. Calculate the accuracy based on the count of correct labels

### **Expected testing accuracy is above 80%**

In [13]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for data, target in test_loader:
        data, target = data.to(device), target.to(device)
        output = model(data)
        _, predicted = torch.max(output, 1)
        correct += (predicted == target).sum().item()
        total += target.size(0)

test_accuracy = 100.0 * correct / total
print(f"Test Accuracy: {test_accuracy:.2f}%")


Test Accuracy: 85.68%


### e) Save and download your model (2 Marks)

**Save your model trained on studio data**

* Save the state dictionary of the classifier (use pytorch only), It will be useful in
integrating model to the web application

 [Hint](https://pytorch.org/tutorials/beginner/saving_loading_models.html)

In [14]:
# Save in the format Order.py expects on the server:
#   ckpt = torch.load(".../speech_model.t7")
#   model.load_state_dict(ckpt['net_dict'])
# A bare state_dict() alone would NOT match -- it has to be wrapped in a
# dict under the 'net_dict' key.
model_path = 'studio_model.t7'
torch.save({'net_dict': model.state_dict()}, model_path)
print(f"Model saved to {model_path}")
print("NOTE: rename this to 'speech_model.t7' when uploading to the server, "
      "that's the exact filename Order.py looks for.")


Model saved to studio_model.t7
NOTE: rename this to 'speech_model.t7' when uploading to the server, that's the exact filename Order.py looks for.


Download your trained model using the code below
* Give the path of model file to download through the browser

In [15]:
from google.colab import files
files.download(model_path)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### f) Deploy and evaluate your model trained on Studio Data in the server (6 Marks).

(This can be done on the day of the Hackathon once the login username and password provided by the mentors in the lab)

Deploy your model on the server, check the hackathon document (2-Server Access and File transfer For Voice based e-commerce ordering.pdf) for details.

To order product in user interface, go through the document (3-Hackathon_II Application Interface Documentation.pdf) for details.


**Evaluation Criteria: Four consecutive utterances should be predicted correctly by the model**

- There are two stages in the e-commerce ordering application    
    - Ordering Product
    - Selecting the e-commerce platform
- If both the stages are cleared as per the evaluation criteria you will get
complete marks Otherwise, you will see a reduction in the marks

## **Stage 2:** Collect your voice samples and refine the classifier trained on studio_data and Team_data
---

### a) Collect your Team Voice Samples and extract features (6 Marks)

(This can be done on the day of the Hackathon once the login username and password is given by mentors in the lab)

* In order to collect the team data, ensure the server is active (2-Server Access and File transfer For Voice based e-commerce ordering.pdf)

* Refer document "3-Hackathon_II Application Interface Documentation.pdf" for collecting your team voice samples. These will get stored in your server

**Evaluation Criteria:**
* Load 'Team_data' and extract features
* Combine features of team data with the extracted features of studio data
* Split the combined features into train and test data
* Load the dataset with DataLoader

In [ ]:
!mkdir team_data

In [ ]:
# Replace <YOUR_GROUP_ID> with your Username given in the lab
!wget -r -A .wav https://aiml-sandbox1.talentsprint.com/audio_recorder/<YOUR_GROUP_ID>/team_data/ -nH --cut-dirs=100  -P ./team_data

In [ ]:
team_features, team_labels = load_data('./team_data')
print("Team data shape:", team_features.shape)


In [ ]:
combined_features = np.vstack((studio_recorded_features, team_features))
combined_labels = np.concatenate((studio_recorded_labels, team_labels))

print("Combined features shape:", combined_features.shape)
print("Combined labels shape:", combined_labels.shape)


In [ ]:
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    combined_features,
    combined_labels,
    test_size=0.2,
    random_state=42,
    stratify=combined_labels
)

print("Train shape:", X_train_c.shape, "Test shape:", X_test_c.shape)


In [ ]:
X_train_c_tensor = torch.from_numpy(X_train_c).float().unsqueeze(2)
y_train_c_tensor = torch.from_numpy(y_train_c).long()

X_test_c_tensor = torch.from_numpy(X_test_c).float().unsqueeze(2)
y_test_c_tensor = torch.from_numpy(y_test_c).long()

train_dataset_c = TensorDataset(X_train_c_tensor, y_train_c_tensor)
test_dataset_c = TensorDataset(X_test_c_tensor, y_test_c_tensor)

batch_size = 16
train_loader_c = DataLoader(train_dataset_c, batch_size=batch_size, shuffle=True)
test_loader_c = DataLoader(test_dataset_c, batch_size=batch_size, shuffle=False)

print("Batches per epoch (train):", len(train_loader_c))


### b) Classify and download the model (6 Marks)

The goal here is to train and test your model on all voice samples collected in studio and team data

**Evaluation Criteria:**
* Refine your classifier (if needed)
* Train your model on the extracted train data
* Test your model on the extracted test data
* Save and download the trained model

### **Expected testing accuracy is above 80%**

In [ ]:
# Start from the studio-trained weights and fine-tune on studio+team data
# with a smaller learning rate, so the model adapts without forgetting
# what it already learned.
model.load_state_dict(torch.load(model_path, map_location=device))
model = model.to(device)

criterion = nn.NLLLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)   # smaller LR for fine-tuning


In [ ]:
num_epochs = 20

train_losses_c = []
train_accuracies_c = []

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for data, target in train_loader_c:
        data, target = data.to(device), target.to(device)

        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * data.size(0)
        _, predicted = torch.max(output, 1)
        correct += (predicted == target).sum().item()
        total += target.size(0)

    epoch_loss = running_loss / total
    epoch_acc = 100.0 * correct / total
    train_losses_c.append(epoch_loss)
    train_accuracies_c.append(epoch_acc)

    print(f"Epoch [{epoch+1}/{num_epochs}]  Loss: {epoch_loss:.4f}  Train Accuracy: {epoch_acc:.2f}%")


In [ ]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for data, target in test_loader_c:
        data, target = data.to(device), target.to(device)
        output = model(data)
        _, predicted = torch.max(output, 1)
        correct += (predicted == target).sum().item()
        total += target.size(0)

test_accuracy_c = 100.0 * correct / total
print(f"Test Accuracy (studio + team data): {test_accuracy_c:.2f}%")


**Save your trained model**

* Save the state dictionary of the classifier (use pytorch only), It will be useful in
integrating model to the web application

 [Hint](https://pytorch.org/tutorials/beginner/saving_loading_models.html)

In [ ]:
team_model_path = 'studio_team_model.t7'
torch.save({'net_dict': model.state_dict()}, team_model_path)
print(f"Model saved to {team_model_path}")
print("NOTE: rename this to 'speech_model.t7' when uploading to the server, "
      "that's the exact filename Order.py looks for.")


Download your trained model using the code below
* Give the path of model file to download through the browser

In [ ]:
from google.colab import files
files.download(team_model_path)


### c) Deploy and evaluate your model trained on Studio Data + Team Data in the server (6 Marks).

(This can be done on the day of the Hackathon once the login username and password provided by the mentors in the lab)

Deploy your model on the server, check the hackathon document (2-Server Access and File transfer For Voice based e-commerce ordering.pdf) for details.

To order product in user interface, go through the document (3-Hackathon_II Application Interface Documentation.pdf) for details.


**Evaluation Criteria: Four consecutive utterances should be predicted correctly by the model**

- There are two stages in the e-commerce ordering application    
    - Ordering Product
    - Selecting the e-commerce platform
- If both the stages are cleared as per the evaluation criteria you will get
complete marks Otherwise, you will see a reduction in the marks